In [272]:
import sys
from pathlib import Path
import pandas as pd 
import numpy as np

sys.path.append(str(Path.cwd().parent))

from src.data_loading import load_all_stations
from src.cleaning import replace_missing_values, convert_units

raw = load_all_stations()
clean = replace_missing_values(raw)
cleaned = convert_units(clean)
print('Raw shape:', raw.shape)
print('Columns:', raw.columns.to_list())
print(raw.head(3))


Raw shape: (61130, 28)
Columns: ['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'TEMP', 'TEMP_ATTRIBUTES', 'DEWP', 'DEWP_ATTRIBUTES', 'SLP', 'SLP_ATTRIBUTES', 'STP', 'STP_ATTRIBUTES', 'VISIB', 'VISIB_ATTRIBUTES', 'WDSP', 'WDSP_ATTRIBUTES', 'MXSPD', 'GUST', 'MAX', 'MAX_ATTRIBUTES', 'MIN', 'MIN_ATTRIBUTES', 'PRCP', 'PRCP_ATTRIBUTES', 'SNDP', 'FRSHTT']
       STATION       DATE   LATITUDE  LONGITUDE  ELEVATION  \
0  41184099999 1996-01-01  25.613483  55.938817      31.08   
1  41184099999 1996-01-02  25.613483  55.938817      31.08   
2  41184099999 1996-01-03  25.613483  55.938817      31.08   

                               NAME  TEMP  TEMP_ATTRIBUTES  DEWP  \
0  RAS AL KHAIMAH INTERNATIONAL, AE  68.8               22  58.7   
1  RAS AL KHAIMAH INTERNATIONAL, AE  70.4               21  60.4   
2  RAS AL KHAIMAH INTERNATIONAL, AE  70.0               23  59.5   

   DEWP_ATTRIBUTES  ...  MXSPD   GUST   MAX  MAX_ATTRIBUTES   MIN  \
0               21  ...    7.0  999.9  

61,130 daily records across six stations. Raw shape before adding derived notebook columns: (61130, 28).

In [273]:
print(f"Rows per station:\n{raw['NAME'].value_counts()}")
print(f"Earliest date for each station:\n{raw.groupby('NAME')['DATE'].min()}")
print(f"Latest date for each station:\n{raw.groupby('NAME')['DATE'].max()}")

Rows per station:
NAME
ABU DHABI INTERNATIONAL, AE         10219
DUBAI INTERNATIONAL, AE             10216
SHARJAH INTERNATIONAL, AE           10205
AL AIN INTERNATIONAL, AE            10192
FUJAIRAH INTERNATIONAL, AE          10161
RAS AL KHAIMAH INTERNATIONAL, AE    10137
Name: count, dtype: int64
Earliest date for each station:
NAME
ABU DHABI INTERNATIONAL, AE        1996-01-01
AL AIN INTERNATIONAL, AE           1996-01-01
DUBAI INTERNATIONAL, AE            1996-01-01
FUJAIRAH INTERNATIONAL, AE         1996-01-01
RAS AL KHAIMAH INTERNATIONAL, AE   1996-01-01
SHARJAH INTERNATIONAL, AE          1996-01-01
Name: DATE, dtype: datetime64[us]
Latest date for each station:
NAME
ABU DHABI INTERNATIONAL, AE        2023-12-31
AL AIN INTERNATIONAL, AE           2023-12-31
DUBAI INTERNATIONAL, AE            2023-12-31
FUJAIRAH INTERNATIONAL, AE         2023-12-31
RAS AL KHAIMAH INTERNATIONAL, AE   2023-12-31
SHARJAH INTERNATIONAL, AE          2023-12-31
Name: DATE, dtype: datetime64[us]


In [274]:
print(f"Number of days between earliest and latest date for each station:\n{raw.groupby('NAME')['DATE'].apply(lambda x: (x.max() - x.min()).days)}")

Number of days between earliest and latest date for each station:
NAME
ABU DHABI INTERNATIONAL, AE         10226
AL AIN INTERNATIONAL, AE            10226
DUBAI INTERNATIONAL, AE             10226
FUJAIRAH INTERNATIONAL, AE          10226
RAS AL KHAIMAH INTERNATIONAL, AE    10226
SHARJAH INTERNATIONAL, AE           10226
Name: DATE, dtype: int64


In [275]:
calendar = pd.date_range('1996-01-01', '2023-12-31', freq='D')
coverage = raw.groupby('NAME')['DATE'].apply(lambda dates: len(calendar.difference(pd.DatetimeIndex(dates))))
print('Missing calendar days per station:')
print(coverage)
print('Total missing days:', coverage.sum())
print('Expected rows:', len(calendar) * raw['NAME'].nunique())

Missing calendar days per station:
NAME
ABU DHABI INTERNATIONAL, AE          8
AL AIN INTERNATIONAL, AE            35
DUBAI INTERNATIONAL, AE             11
FUJAIRAH INTERNATIONAL, AE          66
RAS AL KHAIMAH INTERNATIONAL, AE    90
SHARJAH INTERNATIONAL, AE           22
Name: DATE, dtype: int64
Total missing days: 232
Expected rows: 61362


Inspecting Ras Al Khaimah gaps

In [276]:
dates = pd.date_range('1996-01-01', '2023-12-31', freq='D')
rak = raw[raw['STATION'] == 41184099999]
present = pd.DatetimeIndex(rak["DATE"])

missing_dates = dates.difference(present)
print(f"missing days from rak: {len(missing_dates)}")

missing_by_month = pd.Series(missing_dates).dt.to_period("M").value_counts().sort_index()
print(f"Missing by Month: \n{missing_by_month}")


missing days from rak: 10227
Missing by Month: 
1996-01    31
1996-02    29
1996-03    31
1996-04    30
1996-05    31
           ..
2023-08    31
2023-09    30
2023-10    31
2023-11    30
2023-12    31
Freq: M, Name: count, Length: 336, dtype: int64


Checking Monthly precipitation completeness (D3)

In [277]:
monthly = cleaned.assign(MONTH=cleaned['DATE'].dt.to_period('M'))
months = pd.period_range('1996-01', '2023-12', freq='M')
full_index = pd.MultiIndex.from_product(
    [sorted(monthly['NAME'].unique()), months], names=['NAME', 'MONTH']
)
valid_days = monthly.groupby(['NAME', 'MONTH'])['PRCP'].count().reindex(full_index, fill_value=0)
print('Station-months:', len(valid_days))
for threshold in [28, 25, 20]:
    print(f'Below {threshold} valid days:', (valid_days < threshold).sum())
print('Completely absent months:', (valid_days == 0).sum())

Station-months: 2016
Below 28 valid days: 127
Below 25 valid days: 24
Below 20 valid days: 4
Completely absent months: 1


Decision D3 : Require at least 28 valid PRCP days to retain a monthly rainfall target.

In [278]:
print(raw[['STATION', 'FRSHTT']].dtypes)
print(raw['FRSHTT'].head())

STATION    str
FRSHTT     str
dtype: object
0    000000
1    010000
2    000000
3    010000
4    000000
Name: FRSHTT, dtype: str


Overall missingness after cleaning

In [279]:
missing_pct = (cleaned.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.to_string())

GUST                93.214461
SLP                 17.147064
PRCP                 1.349583
MXSPD                0.171765
VISIB                0.104695
MAX                  0.004908
MIN                  0.004908
DEWP                 0.001636
LONGITUDE            0.000000
LATITUDE             0.000000
STATION              0.000000
DATE                 0.000000
SLP_ATTRIBUTES       0.000000
DEWP_ATTRIBUTES      0.000000
TEMP                 0.000000
TEMP_ATTRIBUTES      0.000000
NAME                 0.000000
ELEVATION            0.000000
VISIB_ATTRIBUTES     0.000000
STP                  0.000000
WDSP                 0.000000
WDSP_ATTRIBUTES      0.000000
STP_ATTRIBUTES       0.000000
MAX_ATTRIBUTES       0.000000
MIN_ATTRIBUTES       0.000000
PRCP_ATTRIBUTES      0.000000
SNDP                 0.000000
FRSHTT               0.000000


In [280]:
slp_by_station = cleaned.groupby('NAME')['SLP'].apply(lambda x: x.isna().mean() * 100)
print('SLP missing percentage by station:')
print(slp_by_station.sort_values(ascending=False).to_string())

slp_by_year = cleaned.assign(YEAR=cleaned['DATE'].dt.year).groupby(['NAME', 'YEAR'])['SLP'].apply(
    lambda x: x.isna().mean() * 100
)
print('Station-years with 100% missing SLP:')
print(slp_by_year[slp_by_year == 100].to_string())

SLP missing percentage by station:
NAME
FUJAIRAH INTERNATIONAL, AE          44.631434
AL AIN INTERNATIONAL, AE            22.556907
DUBAI INTERNATIONAL, AE             12.852388
RAS AL KHAIMAH INTERNATIONAL, AE     8.197692
SHARJAH INTERNATIONAL, AE            8.123469
ABU DHABI INTERNATIONAL, AE          6.605343
Station-years with 100% missing SLP:
NAME                        YEAR
AL AIN INTERNATIONAL, AE    2001    100.0
                            2002    100.0
                            2003    100.0
FUJAIRAH INTERNATIONAL, AE  1996    100.0
                            1997    100.0
                            1998    100.0
                            1999    100.0
                            2000    100.0
                            2001    100.0
                            2002    100.0
                            2003    100.0
                            2022    100.0


Decision D6: Exclude SLP from the main feature set because long, station-dependent gaps would require extensive imputation and complicate cross-station comparisons.

Next: Calculating zero-rainfall percentages

Zero-rainfall % = (Days with PRCP = 0) / (Days with valid PRCP ) * 100


In [281]:
zero_rainfall_prc = cleaned.groupby("NAME")["PRCP"].apply(
    lambda x: (x.eq(0).sum() / x.notna().sum()) *100
)
print(zero_rainfall_prc.sort_values(ascending=False).round(2))

NAME
AL AIN INTERNATIONAL, AE            98.32
ABU DHABI INTERNATIONAL, AE         98.14
FUJAIRAH INTERNATIONAL, AE          97.38
RAS AL KHAIMAH INTERNATIONAL, AE    97.25
DUBAI INTERNATIONAL, AE             97.20
SHARJAH INTERNATIONAL, AE           97.15
Name: PRCP, dtype: float64


counting precipitation flags and analysis

the flags describe how the precipitation measurement was reported or accumlated.

In [282]:
Prcp_attr_count = cleaned['PRCP_ATTRIBUTES'].value_counts()
print(Prcp_attr_count)

PRCP_ATTRIBUTES
I    35481
G     9779
B     5396
D     4568
E     1894
C     1191
A     1164
       825
H      446
F      386
Name: count, dtype: int64


In [283]:
Flags = ['A','B','C','D','E','F','G','H','I',' ']
Meaning = ['One report of 6-h prcp amount', 'Sum of two reports of 6-hour prcp amount(12 hours)','Sum of three 6-hour reports (18 hours)',
           'Sum of four 6-hour reports (24 hours)','One 12-hour precipitation report','Sum of two 12-hour reports (24 hours)','One 24-hour precipitation report',
            'Reported zero rainfall, but precipitation was observed at least once; possible trace or incomplete reporting', 'No precipitation amount reported and no precipitation occurrence reported; rainfall may still have occurred','']

Flag_Meaning = {
    'Flags': Flags,
    'Meaning': Meaning
}
print(f"{"Flag":<8}{"Count":<8}{"Meaning"}")

for count, (flag, meaning) in enumerate(zip(Flag_Meaning['Flags'], Flag_Meaning['Meaning']), start=1):
    print(f"{flag:<8}{Prcp_attr_count[flag]:<8}{meaning}")


Flag    Count   Meaning
A       1164    One report of 6-h prcp amount
B       5396    Sum of two reports of 6-hour prcp amount(12 hours)
C       1191    Sum of three 6-hour reports (18 hours)
D       4568    Sum of four 6-hour reports (24 hours)
E       1894    One 12-hour precipitation report
F       386     Sum of two 12-hour reports (24 hours)
G       9779    One 24-hour precipitation report
H       446     Reported zero rainfall, but precipitation was observed at least once; possible trace or incomplete reporting
I       35481   No precipitation amount reported and no precipitation occurrence reported; rainfall may still have occurred
        825     


In [284]:
cleaned.loc[
    cleaned["PRCP_ATTRIBUTES"] == "I",
    "PRCP"
].value_counts(dropna=False).head(10)

PRCP
0.0    35481
Name: count, dtype: int64

In [285]:
cleaned.groupby("PRCP_ATTRIBUTES")["PRCP"].apply(
    lambda x: (x > 0).sum()
).sort_values(ascending=False)

PRCP_ATTRIBUTES
E    591
G    504
F    292
D     42
A     20
C     10
B      5
       0
H      0
I      0
Name: PRCP, dtype: int64

- Rainfall is not evenly distributed across flags
- Flag I is approximately 58% of the dataset which means no PRCP was reported in these days

so i'll investigate the rainfall amounts for each flag

In [286]:
cleaned.groupby("PRCP_ATTRIBUTES")["PRCP"].max()


PRCP_ATTRIBUTES
         NaN
A     55.880
B     12.954
C     40.132
D     80.010
E    109.982
F    101.092
G    462.534
H      0.000
I      0.000
Name: PRCP, dtype: float64

In [287]:
cleaned.loc[
    cleaned['PRCP'].idxmax()
]

STATION                            41198099999
DATE                       2014-01-19 00:00:00
LATITUDE                             25.112225
LONGITUDE                            56.323964
ELEVATION                                46.32
NAME                FUJAIRAH INTERNATIONAL, AE
TEMP                                 19.611111
TEMP_ATTRIBUTES                             24
DEWP                                 12.111111
DEWP_ATTRIBUTES                             24
SLP                                     1018.3
SLP_ATTRIBUTES                               7
STP                                      999.9
STP_ATTRIBUTES                               0
VISIB                                 8.529502
VISIB_ATTRIBUTES                            23
WDSP                                   13.3344
WDSP_ATTRIBUTES                             23
MXSPD                                  35.9288
GUST                                       NaN
MAX                                  23.722222
MAX_ATTRIBUTE

this observation looks extreme and there's two hypothesis that could address this: 
1.  A severe storm produced this heavy precipitation in Fujairah at that day.
2. The precipitation amount was incorrectly recorded.

To determine which one i'll investigate the surronding days.

In [288]:
mask = cleaned['DATE'].between("2014-01-16", "2014-01-22")
station_mask = cleaned['STATION'] == '41198099999'
columns = ['DATE', 'PRCP', 'PRCP_ATTRIBUTES', 'FRSHTT', 'WDSP']
print(cleaned.loc[mask & station_mask, columns])

            DATE     PRCP PRCP_ATTRIBUTES  FRSHTT     WDSP
37092 2014-01-16    0.000               I  000000  20.0016
37093 2014-01-17    0.000               I  000000  10.3712
37094 2014-01-18    0.000               I  000000   9.4452
37095 2014-01-19  462.534               G  010010  13.3344
37096 2014-01-20    4.064               G  000000  13.1492
37097 2014-01-21    0.000               G  000000  10.1860
37098 2014-01-22    0.000               I  000000  10.0008


Since the extreme PRCP is in Fujairah in in one day in that week, i'll investigate the other stations to check their's as well

In [289]:
mask = cleaned['DATE'].between("2014-01-16", "2014-01-22")
station_mask = cleaned['STATION'] 
columns = ['STATION','DATE', 'PRCP', 'PRCP_ATTRIBUTES', 'FRSHTT', 'WDSP']
print(cleaned.loc[mask & station_mask, columns])

           STATION       DATE     PRCP PRCP_ATTRIBUTES  FRSHTT     WDSP
6504   41184099999 2014-01-16    0.000               I  000000   9.8156
6505   41184099999 2014-01-17    0.000               I  000000   5.1856
6506   41184099999 2014-01-18    0.000               I  000000  12.7788
6507   41184099999 2014-01-19    0.000               E  010000  13.1492
6508   41184099999 2014-01-20    2.032               F  010000   9.8156
6509   41184099999 2014-01-21    0.000               I  000000   7.2228
6510   41184099999 2014-01-22    0.000               I  000000   6.6672
16725  41194099999 2014-01-16    0.000               G  000000  11.1120
16726  41194099999 2014-01-17    0.000               G  000000  10.7416
16727  41194099999 2014-01-18    0.000               G  000000  12.0380
16728  41194099999 2014-01-19    0.000               G  010000  14.8160
16729  41194099999 2014-01-20    0.000               G  000000  10.1860
16730  41194099999 2014-01-21    0.000               G  000000  

C:\Users\marya\AppData\Local\Temp\ipykernel_28108\4075829254.py:4: Pandas4Warning: 'and' operations between boolean dtype and str are deprecated and will raise in a future version. Explicitly cast the strings to a boolean dtype before operating instead.
  print(cleaned.loc[mask & station_mask, columns])


The important Findings:
- there is evidenve of a regional weather event. Five of the six stations recorded a rain indicator on Jan 19.
- Fujairah is an extreme spatial outlier, but rainfall can vary dramatically between locations,especially mountainous terrain.


so this observation would be flagged for sensitivity analysis to evaluate the effect if including or excluding it from the expermint.

# Investigating repeated values

In [290]:
rain = np.isclose(cleaned["PRCP"], 59.944)
attr = cleaned['PRCP_ATTRIBUTES'] == "G"
columns = ['NAME', 'DATE', 'PRCP', 'PRCP_ATTRIBUTES','FRSHTT']
print(cleaned.loc[rain & attr, columns])

                                   NAME       DATE    PRCP PRCP_ATTRIBUTES  \
3836   RAS AL KHAIMAH INTERNATIONAL, AE 2006-09-27  59.944               G   
4521   RAS AL KHAIMAH INTERNATIONAL, AE 2008-08-12  59.944               G   
5912   RAS AL KHAIMAH INTERNATIONAL, AE 2012-06-03  59.944               G   
7232   RAS AL KHAIMAH INTERNATIONAL, AE 2016-01-14  59.944               G   
7488   RAS AL KHAIMAH INTERNATIONAL, AE 2016-09-26  59.944               G   
7899   RAS AL KHAIMAH INTERNATIONAL, AE 2017-11-11  59.944               G   
9653   RAS AL KHAIMAH INTERNATIONAL, AE 2022-09-04  59.944               G   
9666   RAS AL KHAIMAH INTERNATIONAL, AE 2022-09-17  59.944               G   
10132  RAS AL KHAIMAH INTERNATIONAL, AE 2023-12-27  59.944               G   
24598         SHARJAH INTERNATIONAL, AE 2007-09-03  59.944               G   
24815         SHARJAH INTERNATIONAL, AE 2008-04-07  59.944               G   
26334         SHARJAH INTERNATIONAL, AE 2012-06-04  59.944      

investigating RAK, Sep 2022

In [291]:
station = cleaned['STATION'] == '41184099999'

month = cleaned['DATE'].between(
    '2022-09-01',
    '2022-09-30'
)

sep = cleaned.loc[station & month, columns]
monthly_total = sep['PRCP'].sum()

print('Monthly rainfall:', monthly_total.round(2), 'mm')

valid_prcp = sep['PRCP'].count()
print("Valid precipitation observations:", valid_prcp)
print(f"Contribution (%) = {(119.888/monthly_total * 100).round(2)}")

Monthly rainfall: 119.89 mm
Valid precipitation observations: 30
Contribution (%) = 100.0


this means the all recorded rainfall for sep 2022 comes from the two suspicious observations

Objective : Identify all station-month that fail the D3 completeness requirement

In [292]:
cleaned['MONTH'] = cleaned['DATE'].dt.to_period("M")

monthly_counts = (
    cleaned.groupby(['STATION','MONTH']).agg(
        PRCP_VALID_DAYS=("PRCP", 'count')
    )
    .reset_index()
)
print(monthly_counts)

          STATION    MONTH  PRCP_VALID_DAYS
0     41184099999  1996-01               29
1     41184099999  1996-02               27
2     41184099999  1996-03               28
3     41184099999  1996-04               30
4     41184099999  1996-05               31
...           ...      ...              ...
2010  41218099999  2023-08               31
2011  41218099999  2023-09               30
2012  41218099999  2023-10               30
2013  41218099999  2023-11               28
2014  41218099999  2023-12               31

[2015 rows x 3 columns]


In [293]:
incomplete_months = monthly_counts[
    monthly_counts['PRCP_VALID_DAYS'] < 28
]

print("Incomplete months:", len(incomplete_months))
print(incomplete_months.head(10))

Incomplete months: 126
         STATION    MONTH  PRCP_VALID_DAYS
1    41184099999  1996-02               27
21   41184099999  1997-10               22
25   41184099999  1998-02               27
36   41184099999  1999-01               26
37   41184099999  1999-02               25
38   41184099999  1999-03               27
73   41184099999  2002-02               27
85   41184099999  2003-02               27
119  41184099999  2005-12                6
120  41184099999  2006-02               13


Objective

Construct a complete monthly calendar for all six UAE weather stations between January 1996 and December 2023, including months with no recorded observations.

In [294]:
all_months = pd.period_range(
    start='1996-01',
    end='2023-12',
    freq='M'
)
stations = cleaned['STATION'].unique()

full_index = pd.MultiIndex.from_product(
    [stations, all_months],
    names=['STATION','MONTH']
)

complete_monthly = (
    monthly_counts.set_index(['STATION','MONTH']).reindex(full_index).reset_index()
)

print("Total station-months:", len(complete_monthly))

print(complete_monthly[
    (complete_monthly["STATION"] == '41184099999') &
    (complete_monthly['MONTH'] == pd.Period('2006-01', freq='M'))
])


Total station-months: 2016
         STATION    MONTH  PRCP_VALID_DAYS
120  41184099999  2006-01              NaN


The completed monthly calendar contains 2016 station-month combinations. January 2006 record was seccessfully restored with NaN.

Entirely missing station-months are preserved in the monthly calendar with missing values rather than replaced with historical averages. This avoids introducing synthetic ground-truth rainfall measurements and potentially biasing model evaluation.

NEXT: identifying Incomplete Station-Months. 

Identify station-months with insufficient daily precipitation observations based on the D3 completeness criterion.

In [295]:
incomplete_mask = (
    (complete_monthly['PRCP_VALID_DAYS'] < 28)
    |
    complete_monthly['PRCP_VALID_DAYS'].isna()
)

incomplete_months = complete_monthly.loc[incomplete_mask]

n_incomplete = len(incomplete_months)
n_total = len(complete_monthly)

incomplete_percentage = (n_incomplete/n_total) *100

print("Total station-months:", n_total)
print("Incomplete station-months:", n_incomplete)
print(f"Incomplete percentage: {incomplete_percentage:.2f}%")

Total station-months: 2016
Incomplete station-months: 127
Incomplete percentage: 6.30%


Out of 2,016 station-months, 127 (6.30%) had fewer than 28 days with valid rainfall measurements. The remaining 1,889 (93.70%) met the requirement.

We considered lowering the requirement to 25 days to keep more data. However, missing days could make the monthly rainfall total look lower than it really was, especially if rain fell on those days.

For now, we will keep the 28-day rule. We will later compare different thresholds to see how they affect the data and model results.

If a month does not meet the rainfall requirement, we will mark its rainfall total as missing—not zero. We will not automatically delete its temperature, wind, or other weather measurements, because those may still be useful if they pass their own quality checks.

Next: Completeness Theshold sensitivity analysis

In [296]:
print(f"{"Threshold":<17}{"Valid months":<19}{"Retention%"}")
print("-"*55)

valid_25 = (complete_monthly['PRCP_VALID_DAYS'] >= 25).sum()
ret_25 = (valid_25/n_total) * 100

valid_28 = (complete_monthly['PRCP_VALID_DAYS'] >= 28).sum()
ret_28 = (valid_28/n_total) * 100

valid_30 = (complete_monthly['PRCP_VALID_DAYS'] >= 30).sum()
ret_30 = (valid_30/n_total) * 100

print(f"{"<=25 days":<17}{valid_25:<19}{ret_25.round(2)}")
print(f"{"<=28 days":<17}{valid_28:<19}{ret_28.round(2)}")
print(f"{"<=30 days":<17}{valid_30:<19}{ret_30.round(2)}")



Threshold        Valid months       Retention%
-------------------------------------------------------
<=25 days        1992               98.81
<=28 days        1889               93.7
<=30 days        1579               78.32


#### results:

the retention percentage changed across the three thresholds giving the 25 days threshold the highest retention rate (98.81%) while the 30 days threshold the lowest retention percentage (78.32%). 28-days threshold the intermediate percentage of (93.7%)

looking at the 30-day threshold the result identifies that many months in the data sets have less days than this treshold. so the retention is low.

we should avoid selecting a complete threshold as based on the number of retained months data retention alone does not guarantee reliable rainfall targets

**Investigating Relative Monthly Completeness**

comparing the (monthly completeness >= 90 %) with (valid days >=28) thresholds

I'm investigating a percentage-based completeness criterion to check if it's fairer among varying calendar-month lengths that could yield better retention and to check the precipitation as well.

Relative monthly completeness is calculated by getting the number of days in each month and also the number of valid preceptation days in each month as well. Then dividing these valid prcp days with the total days in month get the completeness percentage of each month related to the PRCP.

Then compared with the 28<= valid precipitation threshold to validate the best retention percentage.

In [297]:
complete_monthly['DAYS_IN_MONTH'] = complete_monthly['MONTH'].dt.days_in_month

complete_monthly['PRCP_COMPLETENESS'] = (complete_monthly['PRCP_VALID_DAYS']/complete_monthly["DAYS_IN_MONTH"]) * 100
columns = ['MONTH','PRCP_VALID_DAYS', 'DAYS_IN_MONTH','PRCP_COMPLETENESS']
print(f"{complete_monthly[columns].head(5)}")

     MONTH  PRCP_VALID_DAYS  DAYS_IN_MONTH  PRCP_COMPLETENESS
0  1996-01             29.0             31          93.548387
1  1996-02             27.0             29          93.103448
2  1996-03             28.0             31          90.322581
3  1996-04             30.0             30         100.000000
4  1996-05             31.0             31         100.000000


In [298]:
Accepted_months = (complete_monthly['PRCP_COMPLETENESS']>=90).sum() 
print("Accepted months:",Accepted_months)
ret_perc = (Accepted_months/n_total) * 100
print("Retention Perc:",ret_perc)
relative_only_months = (
    (complete_monthly['PRCP_COMPLETENESS']>=90)
    &
    (complete_monthly['PRCP_VALID_DAYS'] < 28)
).sum()
print("relative only months:", relative_only_months)

Accepted months: 1941
Retention Perc: 96.27976190476191
relative only months: 52


using the relative >=90% completeness yielded 52 more months more than the 28<= valid days. the retention percentage is higher 96.28%, where the 28<= valid days retention was 93.70%.

This is becasue the relative completeness accepts months where the 28<= threshold rejects, for example February 1996 which had 27 valid days but 29 calendar days. this month is accepted in th the relative completeness critera but rejected in the 28<= threshold

But last validation is to check if there's any accepted month in the 28<= valid days critera that the 90<= completeness rejects. and decide which one i'll go with


In [ ]:
Both_Accepted = (
    (complete_monthly['PRCP_VALID_DAYS'] >= 28)
    &
    (complete_monthly['PRCP_COMPLETENESS'] < 90)
)

print(complete_monthly[Both_Accepted].shape[0])

0


there's no months that the thresholds accept together because 28 days from 31 days is approximatly 90.32%, which is the least valid percentage for a month with 28 valid days.

**Research Interpretation and Decision**

The 90% threshold is fairer because it considers the varying calendar-month lengths. For example, if February has 27 valid days of 29 days, 28 days threshold rejects it but the 90% threshold accepts it. The additional 52 months may not guarantee imporvement in prediction performance as the missing observations may have recorded precipitation amount, therefore the total recorded monthly precipitation amount may be less than the actual total, and could affect the target reliability. For now, i'm keeping the 28 days threshold temporarily and will validate the 90% threshold using LOSO across the same reliable evaluation set.